# Source Separation Decomposition

Visualises the encoder–decoder decomposition from Phase 1 pretraining (`03b_source_sep_pretraining.py`).

For each of the 7 label combinations (single-, dual-, and triple-positive wells), N random examples are shown.  Each example row has:
- **Col 0** — raw input curve (black) + sum of active decoded channels (red dashed) + individual decoded channels (thin coloured lines).
- **Cols 1–3** — decoded curve for each target channel (solid) overlaid on real single-target reference curves (faint) and their mean (dotted).  Absent channels are shown at reduced opacity so suppression is visible.

> **Prerequisite**: run `03b_source_sep_pretraining.py --force_rerun` once after updating the script — decoder weights (`source_sep_decoder_*_weights.weights.h5`) are saved alongside the encoder weights from that point onward.

In [ ]:
import sys
from pathlib import Path
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import matplotlib.pyplot as plt
import joblib

_MULTIPLEX = Path('..').resolve()
sys.path.insert(0, str(_MULTIPLEX))
sys.path.insert(0, str(_MULTIPLEX / 'utils' / 'model_training'))

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import config_multiplex as config
from model_utils_source_sep import build_source_sep_encoder, build_parametric_decoder

## Config

In [ ]:
# ── Experiment ────────────────────────────────────────────────────────
EXP_PATH = Path('/vol/bitbucket/gk225/POC_DDM_datasets/LAB_Multiplex/01_ACA_qdPCR')

# ── Architecture (must match 03b training args) ────────────────────────
D_SHARED = 16
D_TARGET = 10

# ── Visualisation ─────────────────────────────────────────────────────
N_EXAMPLES   = 5    # random wells shown per label combination
N_REF_CURVES = 20   # single-target reference curves overlaid per channel
SEED         = 42

## Load data

In [ ]:
data_path    = EXP_PATH / config.TRAINING_DATA_PATH
data         = joblib.load(data_path)

X_curves     = data['curves']['ori_curves'].astype(np.float32)   # (N, T)
y_binary     = data['label_binarized'].astype(np.int32)          # (N, 3)
all_targets  = data['all_targets']
T            = X_curves.shape[1]
N, n_targets = y_binary.shape

print(f'N={N}, T={T}, targets={all_targets}')
print()
combos, counts = np.unique(y_binary, axis=0, return_counts=True)
for c, n in zip(combos, counts):
    names = '+'.join(all_targets[j] for j in range(n_targets) if c[j])
    print(f'  {names or "Negative":20s}  n={n:,}')

## Load encoder and decoder weights

In [ ]:
enc_path  = EXP_PATH / 'source_sep_encoder_weights.weights.h5'
dec_paths = [EXP_PATH / f'source_sep_decoder_{j}_weights.weights.h5'
             for j in range(n_targets)]

missing = [p for p in [enc_path] + dec_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing weight files:\n'
        + '\n'.join(f'  {p}' for p in missing)
        + '\n\nRun: python 03b_source_sep_pretraining.py --force_rerun [--validate]'
    )

tf.keras.backend.clear_session()
encoder  = build_source_sep_encoder(T, D_SHARED, D_TARGET, n_targets)
decoders = [build_parametric_decoder(D_SHARED, D_TARGET, T_max=T, j=j)
            for j in range(n_targets)]
encoder.load_weights(enc_path)
for j, dec in enumerate(decoders):
    dec.load_weights(dec_paths[j])

print(f'Encoder  params: {encoder.count_params():,}')
for j, dec in enumerate(decoders):
    print(f'Decoder {all_targets[j]:3s} params: {dec.count_params():,}')

## Decode all curves

In [ ]:
def render_sigmoid_np(params):
    T_ = X_curves.shape[1]
    Fm = params[:, 0:1]; Fb = params[:, 1:2]
    Sc = params[:, 2:3]; Cs = params[:, 3:4]; As = params[:, 4:5]
    t  = np.arange(T_, dtype=np.float32)[np.newaxis, :]
    return Fm / (1.0 + np.exp(-Sc * (t - Cs))) ** As + Fb

X_enc    = np.expand_dims(X_curves, -1)                          # (N, T, 1)
Z        = encoder.predict(X_enc, batch_size=512, verbose=1)     # (N, d_sh+n*d_t)

z_shared = Z[:, :D_SHARED]
z_parts  = [Z[:, D_SHARED + j*D_TARGET : D_SHARED + (j+1)*D_TARGET]
            for j in range(n_targets)]

dec_curves = []
for j, dec in enumerate(decoders):
    dec_in   = np.concatenate([z_shared, z_parts[j]], axis=-1).astype(np.float32)
    params_j = dec.predict(dec_in, batch_size=512, verbose=0)    # (N, 5)
    dec_curves.append(render_sigmoid_np(params_j))               # (N, T)

# Active-channel reconstruction per well
recon = sum(y_binary[:, j:j+1] * dec_curves[j] for j in range(n_targets))
mse   = np.mean((X_curves - recon) ** 2, axis=1)
print(f'\nReconstruction MSE  mean={mse.mean():.5f}  '
      f'median={np.median(mse):.5f}  p95={np.percentile(mse, 95):.5f}')

## Reference curve banks (single-target wells)

In [ ]:
rng       = np.random.default_rng(SEED)
ref_banks = []
for j, tgt in enumerate(all_targets):
    mask_j = (y_binary.sum(axis=1) == 1) & (y_binary[:, j] == 1)
    bank_j = X_curves[mask_j]
    idx    = rng.choice(len(bank_j), size=min(N_REF_CURVES, len(bank_j)), replace=False)
    ref_banks.append(bank_j[idx])
    print(f'  {tgt}: {mask_j.sum():,} single-target wells, showing {len(idx)}')

## Visualisation

In [ ]:
COLORS = {t: c for t, c in zip(all_targets, ['#4472C4', '#ED7D31', '#70AD47'])}
t_axis = np.arange(T)


def plot_combo(combo, n_examples=N_EXAMPLES, seed=SEED):
    combo = np.asarray(combo, dtype=np.int32)
    mask  = np.all(y_binary == combo, axis=1)
    idxs  = np.where(mask)[0]
    if len(idxs) == 0:
        print(f'No wells with combo {combo.tolist()}'); return
    rng2 = np.random.default_rng(seed)
    pick = rng2.choice(idxs, size=min(n_examples, len(idxs)), replace=False)

    active = [j for j in range(n_targets) if combo[j]]
    label  = '+'.join(all_targets[j] for j in active) or 'Negative'

    nrows = len(pick)
    ncols = n_targets + 1
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3.5 * ncols, 2.8 * nrows), squeeze=False)
    fig.suptitle(f'{label}  ({len(idxs):,} wells total, showing {nrows})',
                 fontsize=12, fontweight='bold', y=1.01)

    # Column titles
    axes[0, 0].set_title('Original + active sum', fontsize=9)
    for j, tgt in enumerate(all_targets):
        suffix = '' if combo[j] else ' (absent)'
        col    = COLORS[tgt] if combo[j] else 'grey'
        axes[0, j + 1].set_title(f'{tgt}{suffix}', fontsize=9, color=col)

    for row, wi in enumerate(pick):
        # ── Col 0: original + decomposition ───────────────────────
        ax0 = axes[row, 0]
        ax0.plot(t_axis, X_curves[wi], color='black', lw=1.5, label='Original', zorder=5)
        ax0.plot(t_axis, recon[wi],    color='#C00000', lw=1.2, ls='--',
                 label='Active sum', zorder=4)
        for j in range(n_targets):
            ax0.plot(t_axis, dec_curves[j][wi],
                     color=COLORS[all_targets[j]], lw=0.8,
                     alpha=0.75 if combo[j] else 0.25, zorder=3)
        ax0.set_ylabel(f'well {wi}', fontsize=7)
        if row == 0:
            ax0.legend(fontsize=6, loc='upper left', framealpha=0.7)
        ax0.tick_params(labelsize=7)
        ax0.set_xticks(np.arange(0, T, 10))

        # ── Cols 1+: decoded channel j ────────────────────────────
        for j, tgt in enumerate(all_targets):
            ax  = axes[row, j + 1]
            col = COLORS[tgt]
            is_active = bool(combo[j])
            # Reference curves
            for rc in ref_banks[j]:
                ax.plot(t_axis, rc, color=col, lw=0.4, alpha=0.12, zorder=1)
            ax.plot(t_axis, ref_banks[j].mean(axis=0), color=col, lw=1.0,
                    ls=':', alpha=0.55, label='Ref mean', zorder=2)
            # Decoded
            ax.plot(t_axis, dec_curves[j][wi], color=col,
                    lw=1.5 if is_active else 0.8,
                    alpha=1.0 if is_active else 0.4,
                    label='Decoded', zorder=3)
            if row == 0:
                ax.legend(fontsize=6, loc='upper left', framealpha=0.7)
            ax.tick_params(labelsize=7)
            ax.set_xticks(np.arange(0, T, 10))

    for ax in axes[-1]:
        ax.set_xlabel('Cycle', fontsize=8)

    plt.tight_layout()
    plt.show()

print('plot_combo() ready')

### KPC only

In [ ]:
plot_combo([1, 0, 0])

### NDM only

In [ ]:
plot_combo([0, 1, 0])

### VIM only

In [ ]:
plot_combo([0, 0, 1])

### KPC + NDM

In [ ]:
plot_combo([1, 1, 0])

### KPC + VIM

In [ ]:
plot_combo([1, 0, 1])

### NDM + VIM

In [ ]:
plot_combo([0, 1, 1])

### KPC + NDM + VIM

In [ ]:
plot_combo([1, 1, 1])